In [2]:
import torch
import torchvision

from torchvision import models

In [ ]:
# save the model
from datetime import datetime

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

In [3]:
convnext_tiny_cub = models.convnext_tiny(
    weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1
)
convnext_tiny_cub

ConvNeXt(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
    )
    (1): Sequential(
      (0): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=96, out_features=384, bias=True)
          (4): GELU(approximate='none')
          (5): Linear(in_features=384, out_features=96, bias=True)
          (6): Permute()
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=

In [4]:
# replace the last layer with a new one for CUB dataset

convnext_tiny_cub.classifier.pop(-1)
convnext_tiny_cub.classifier.append(torch.nn.Linear(768, 200))
convnext_tiny_cub

ConvNeXt(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
    )
    (1): Sequential(
      (0): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=96, out_features=384, bias=True)
          (4): GELU(approximate='none')
          (5): Linear(in_features=384, out_features=96, bias=True)
          (6): Permute()
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=

In [5]:
# freeze the model features
for param in convnext_tiny_cub.features.parameters():
    param.requires_grad = False

In [6]:
# train the model on CUB dataset

from datasets.cub import get_cub200

train_ds, val_ds = get_cub200(
    "/home/bubuss/source/UJ/ProtoPool/codebook_train/data/cub", 256, 224, 0.1, 0.5
)

train_dl = torch.utils.data.DataLoader(
    train_ds, batch_size=128, shuffle=True, num_workers=8, pin_memory=True
)
val_dl = torch.utils.data.DataLoader(
    val_ds, batch_size=128, shuffle=False, num_workers=8, pin_memory=True
)

/home/bubuss/source/UJ/ProtoPool/codebook_train/datasets/cub.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.data["label"] -= 1
/home/bubuss/source/UJ/ProtoPool/codebook_train/datasets/cub.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.data["label"] -= 1


In [7]:
def train_epoch(model, train_dl, criterion, optimizer):
    model.train()
    for i, (x, y) in enumerate(train_dl):
        x = x.cuda()
        y = y.cuda()

        optimizer.zero_grad()
        y_hat = convnext_tiny_cub(x)
        loss = criterion(y_hat, y)
        loss.backward()
        optimizer.step()

        if i % (len(train_dl) // 10) == 0:
            accuracy = (y_hat.argmax(1) == y).float().mean()
            print(
                f"Iter {i}/{len(train_dl)} Loss {loss.item()}, Accuracy {accuracy.item()}"
            )


def validate_epoch(model, val_dl, criterion) -> tuple[float, float]:
    model.eval()
    with torch.no_grad():
        total_loss = 0
        total_accuracy = 0
        total_samples = 0
        for i, (x, y) in enumerate(val_dl):
            x = x.cuda()
            y = y.cuda()

            y_hat = convnext_tiny_cub(x)
            loss = criterion(y_hat, y)

            total_loss += loss.item()
            total_accuracy += (y_hat.argmax(1) == y).float().sum().item()
            total_samples += y.size(0)

    accuracy = total_accuracy / total_samples
    return total_loss, accuracy

In [8]:
# implement early stopping for accuracy


class EarlyStopping:
    def __init__(self, patience: int):
        self.patience = patience
        self.counter = 0
        self.best_accuracy = None
        self.early_stop = False

    def __call__(self, accuracy: float) -> bool:
        if self.best_accuracy is None:
            self.best_accuracy = accuracy
        elif accuracy < self.best_accuracy:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_accuracy = accuracy
            self.counter = 0

        return self.early_stop

In [ ]:
# trian the classifier

optimizer = torch.optim.Adam(convnext_tiny_cub.classifier.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss()
convnext_tiny_cub.cuda()
early_stopper = EarlyStopping(patience=3)

for epoch in range(300):
    print(f"Epoch {epoch}")
    train_epoch(convnext_tiny_cub, train_dl, criterion, optimizer)
    val_loss, val_accuracy = validate_epoch(convnext_tiny_cub, val_dl, criterion)
    print(f"Val Loss {val_loss}, Val Accuracy {val_accuracy}")

    if early_stopper(val_accuracy):
        print("Early stopping")
        break
    else:
        torch.save(convnext_tiny_cub.state_dict(), f"temp_checkpoint_{timestamp}.pth")

# load the best model
convnext_tiny_cub.load_state_dict(torch.load(f"temp_checkpoint_{timestamp}.pth"))

In [14]:
convnext_tiny_cub.features[-1]

CNBlock(
  (block): Sequential(
    (0): Conv2d(768, 768, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=768)
    (1): Permute()
    (2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
    (3): Linear(in_features=768, out_features=3072, bias=True)
    (4): GELU(approximate='none')
    (5): Linear(in_features=3072, out_features=768, bias=True)
    (6): Permute()
  )
  (stochastic_depth): StochasticDepth(p=0.1, mode=row)
)

In [15]:
# unfreeze the last layer
for param in convnext_tiny_cub.features[-1].parameters():
    param.requires_grad = True

optimizer_last_layer = torch.optim.Adam(convnext_tiny_cub.parameters(), lr=0.001)
early_stopper_last_layer = EarlyStopping(patience=3)

for epoch in range(300):
    print(f"Epoch {epoch}")
    train_epoch(convnext_tiny_cub, train_dl, criterion, optimizer_last_layer)

    val_loss, val_accuracy = validate_epoch(convnext_tiny_cub, val_dl, criterion)
    print(f"Validation Loss {val_loss}, Accuracy {val_accuracy}")
    if early_stopper_last_layer(val_accuracy):
        print("Early stopping")
        break
    else:
        torch.save(convnext_tiny_cub.state_dict(), f"temp_checkpoint_{timestamp}.pth")

# load the best model
convnext_tiny_cub.load_state_dict(torch.load(f"temp_checkpoint_{timestamp}.pth"))

Epoch 0


NameError: name 'criterion' is not defined

In [ ]:
for param in convnext_tiny_cub.parameters():
    param.requires_grad = True

optimizer_whole = torch.optim.Adam(convnext_tiny_cub.parameters(), lr=0.001)
early_stopper_whole = EarlyStopping(patience=3)


for epoch in range(300):
    print(f"Epoch {epoch}")
    train_epoch(convnext_tiny_cub, train_dl, criterion, optimizer_whole)

    val_loss, val_accuracy = validate_epoch(convnext_tiny_cub, val_dl, criterion)
    print(f"Validation Loss {val_loss}, Accuracy {val_accuracy}")
    if early_stopper_whole(val_accuracy):
        print("Early stopping")
        break
    else:
        torch.save(convnext_tiny_cub.state_dict(), f"temp_checkpoint_{timestamp}.pth")

# load the best model
convnext_tiny_cub.load_state_dict(torch.load(f"temp_checkpoint_{timestamp}.pth"))

In [ ]:
torch.save(convnext_tiny_cub.state_dict(), f"convnext_tiny_cub_{timestamp}.pth")